# MiniMax H3 · 本地画布后端

标准首测：4 秒、768p 短边、16:9、50 步、seed 42，原始 FL2VA 权重。先完成原生基线，再测试加速配置。

此 Notebook 含实际部署与测试代码，不含成功运行的输出。目标机器是已观察到的 RTX PRO 6000 Blackwell 96GB / 176.9GB RAM；单卡放置方案必须经过真实生成验证。

权重版本固定为 `42ed227ee7df40d41602854ae760620d6eb651fe`，只下载 FL2VA，约 134.16 GiB。首次从 Hugging Face 下载至本地 SSD，然后逐文件校验并备份到 Drive。以后从 Drive 恢复至 SSD，推理不直接读取 Drive。

In [ ]:
import subprocess, sys, shutil, pathlib, os, json
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],text=True))
print(sys.version)
print(open('/proc/meminfo').read().splitlines()[0])
print('Local disk free GiB:', round(shutil.disk_usage('/content').free / 2**30, 2))

挂载当前账号的 Google Drive。授权页面由你核对账号；Notebook 不切换账号，也不改变文件共享权限。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
h3_files = {'benchmark.py': '"""Measure native H3 submit-to-downloaded-MP4 latency without the canvas UI."""\nimport argparse\nimport json\nimport os\nimport pathlib\nimport re\nimport time\nimport urllib.error\nimport urllib.request\n\nPROMPT = \'At dawn in a pine forest, a red fox walks slowly through thin mist. A low camera tracks forward smoothly. Soft backlight, realistic fur. Gentle wind and birdsong.\'\nPAYLOAD = {\n    \'model\': \'MiniMaxAI/MiniMax-H3\', \'prompt\': PROMPT, \'seconds\': 4,\n    \'task\': \'t2va\', \'conditions\': [],\n    \'target\': {\'short_edge\': 768, \'aspect_ratio\': \'16:9\', \'duration_seconds\': 4.0},\n    \'num_outputs_per_prompt\': 1, \'num_inference_steps\': 50,\n    \'flow_shift\': 12.0, \'audio_flow_shift\': 3.0, \'seed\': 42, \'quality\': \'lossless\',\n}\n\n\ndef request(base, route, token, payload=None):\n    headers = {\'Authorization\': \'Bearer \' + token} if token else {}\n    body = None\n    if payload is not None:\n        headers[\'Content-Type\'] = \'application/json\'\n        body = json.dumps(payload).encode()\n    return urllib.request.urlopen(urllib.request.Request(base + route, data=body, headers=headers), timeout=60)\n\n\ndef measure(base, output, label, token):\n    started = time.perf_counter()\n    with request(base, \'/v1/videos\', token, PAYLOAD) as response:\n        job = json.load(response)\n    submitted = time.perf_counter()\n    job_id = job[\'id\']\n    if not isinstance(job_id, str) or not re.fullmatch(r\'[A-Za-z0-9_-]{1,100}\', job_id):\n        raise ValueError(\'Invalid remote job ID\')\n    while job[\'status\'] != \'completed\':\n        if job[\'status\'] == \'failed\':\n            raise RuntimeError(json.dumps(job.get(\'error\', job), ensure_ascii=False))\n        if time.perf_counter() - started > 1800:\n            raise TimeoutError(\'30 minutes elapsed; remote GPU work may still be running\')\n        time.sleep(.5)\n        with request(base, \'/v1/videos/\' + job_id, token) as response:\n            job = json.load(response)\n    generated = time.perf_counter()\n    target = output / (label + \'.mp4\')\n    temp = target.with_suffix(\'.mp4.part\')\n    try:\n        with request(base, \'/v1/videos/\' + job_id + \'/content\', token) as response, temp.open(\'wb\') as dest:\n            size = 0\n            while chunk := response.read(1024 * 1024):\n                size += len(chunk)\n                if size > 250 * 1024 * 1024:\n                    raise ValueError(\'Video exceeds 250 MiB\')\n                dest.write(chunk)\n        with temp.open(\'rb\') as video:\n            header = video.read(12)\n        if size < 12 or header[4:8] != b\'ftyp\':\n            raise ValueError(\'Response is not a valid MP4 container\')\n        temp.replace(target)\n    except BaseException:\n        temp.unlink(missing_ok=True)\n        raise\n    finished = time.perf_counter()\n    result = {\'label\': label, \'request\': PAYLOAD, \'job\': job,\n              \'submitMs\': round((submitted-started)*1000),\n              \'generationAndPollingMs\': round((generated-submitted)*1000),\n              \'downloadMs\': round((finished-generated)*1000),\n              \'totalMs\': round((finished-started)*1000), \'bytes\': size,\n              \'scope\': \'API submit to complete MP4 download; excludes browser decode\',\n              \'pollIntervalMs\': 500, \'measuredAt\': time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime())}\n    (output / (label + \'.json\')).write_text(json.dumps(result, indent=2), encoding=\'utf8\')\n    print(json.dumps({k: result[k] for k in (\'label\', \'submitMs\', \'generationAndPollingMs\', \'downloadMs\', \'totalMs\', \'bytes\')}))\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--base\', default=\'http://127.0.0.1:30020\')\n    parser.add_argument(\'--output\', default=\'/content/h3-benchmarks\')\n    parser.add_argument(\'--label\', default=\'cold-standard-4s-768p\')\n    parser.add_argument(\'--self-test\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.self_test:\n        import io\n        import tempfile\n        from unittest.mock import patch\n        with tempfile.TemporaryDirectory() as directory:\n            output = pathlib.Path(directory)\n            with patch(__name__ + \'.request\', side_effect=[\n                io.BytesIO(b\'{"id":"video_1","status":"completed"}\'), io.BytesIO(b\'<html>error</html>\')\n            ]):\n                try:\n                    measure(\'http://mock\', output, \'invalid\', \'\')\n                    raise AssertionError(\'HTML must not produce a completed benchmark\')\n                except ValueError:\n                    assert not list(output.iterdir())\n            with patch(__name__ + \'.request\', return_value=io.BytesIO(b\'{"id":"../../secret","status":"completed"}\')):\n                try:\n                    measure(\'http://mock\', output, \'invalid-id\', \'\')\n                    raise AssertionError(\'Unsafe job ID must be rejected\')\n                except ValueError:\n                    assert not list(output.iterdir())\n        print(\'benchmark failure-path checks passed (no GPU inference)\')\n    else:\n        output = pathlib.Path(args.output)\n        output.mkdir(parents=True, exist_ok=True)\n        if not re.fullmatch(r\'[A-Za-z0-9_-]{1,100}\', args.label):\n            raise SystemExit(\'Label must use only letters, digits, hyphens, or underscores\')\n        measure(args.base.rstrip(\'/\'), output, args.label, os.environ.get(\'H3_PROXY_TOKEN\', \'\'))\n', 'install.sh': '#!/usr/bin/env bash\nset -euo pipefail\npython -m pip install -q uv\nif [ ! -x /content/h3-venv/bin/python ]; then\n  uv venv --python 3.12 /content/h3-venv\nfi\nexport UV_NO_CACHE=1\n# The official source supports disabling optional SRT Rust extensions.\nexport SGLANG_BUILD_RUST_EXTS=none\nuv pip install --python /content/h3-venv/bin/python --prerelease=allow \\\n  \'sglang[diffusion] @ git+https://github.com/sgl-project/sglang.git@45252c64ca8577fb842f953d57f85509a0aa104e#subdirectory=python\'\nif ! command -v ffmpeg >/dev/null || ! command -v ffprobe >/dev/null; then\n  apt-get update -qq\n  apt-get install -y -qq ffmpeg\nfi\n/content/h3-venv/bin/sglang serve --model-type diffusion --help > /content/h3-serve-help.txt\nuv pip freeze --python /content/h3-venv/bin/python > /content/h3-environment.txt\n/content/h3-venv/bin/python "$(dirname "$0")/validate_install.py"\n', 'validate_install.py': '"""Fail before weight loading if required native H3 code or CLI flags are absent."""\nimport importlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\n\ndef main():\n    for executable in (\'ffmpeg\', \'ffprobe\', \'nvidia-smi\'):\n        if not shutil.which(executable):\n            raise RuntimeError(\'Missing executable: \' + executable)\n    for module in (\'sglang.multimodal_gen.runtime.pipelines.minimax_h3_pipeline\',\n                   \'sglang.multimodal_gen.runtime.entrypoints.http_server\',\n                   \'fastapi\', \'httpx\', \'uvicorn\'):\n        importlib.import_module(module)\n    cli = Path(sys.executable).parent/\'sglang\'\n    help_text = subprocess.check_output([str(cli), \'serve\', \'--model-type\', \'diffusion\', \'--help\'], text=True, stderr=subprocess.STDOUT)\n    for flag in (\'--component-residency\', \'--model-variant\', \'--pipeline-class-name\', \'--served-model-name\'):\n        if flag not in help_text:\n            raise RuntimeError(\'Installed CLI is missing \' + flag)\n    hardware = subprocess.check_output([\'nvidia-smi\', \'--query-gpu=name,memory.total,driver_version\',\n                                        \'--format=csv,noheader\'], text=True).strip()\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'PyTorch cannot access the GPU\')\n    record = {\'gpu\': hardware, \'python\': sys.version, \'torch\': torch.__version__,\n              \'torch_cuda\': torch.version.cuda, \'device_capability\': torch.cuda.get_device_capability(),\n              \'sglang_commit\': \'45252c64ca8577fb842f953d57f85509a0aa104e\',\n              \'scope\': \'imports, CLI flags and GPU visibility only; no model inference\'}\n    Path(\'/content/h3-install-check.json\').write_text(json.dumps(record, indent=2))\n    Path(\'/content/h3-install.ok\').touch()\n    print(json.dumps(record, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'cache_model.py': '"""Pin the native FL2VA partition, use local SSD, and keep a verified Drive copy."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport tempfile\n\nREPO = \'MiniMaxAI/MiniMax-H3\'\nREVISION = \'42ed227ee7df40d41602854ae760620d6eb651fe\'\n\n\ndef copy_verified(source, target, expected_size, expected_sha=None):\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temp = target.with_name(target.name + \'.part\')\n    digest = hashlib.sha256()\n    size = 0\n    try:\n        with source.open(\'rb\') as src, temp.open(\'wb\') as dst:\n            while chunk := src.read(8 * 1024 * 1024):\n                size += len(chunk)\n                digest.update(chunk)\n                dst.write(chunk)\n        actual_sha = digest.hexdigest()\n        if size != expected_size or (expected_sha and actual_sha != expected_sha):\n            raise ValueError(\'Model cache checksum mismatch: \' + str(source))\n        temp.replace(target)\n        return actual_sha\n    except BaseException:\n        temp.unlink(missing_ok=True)\n        raise\n\n\ndef cache(local, drive):\n    from huggingface_hub import HfApi, snapshot_download, hf_hub_download\n    if not Path(\'/content/drive/MyDrive\').is_dir():\n        raise RuntimeError(\'Mount the intended Google Drive before caching weights\')\n    local.mkdir(parents=True, exist_ok=True)\n    drive.mkdir(parents=True, exist_ok=True)\n    api = HfApi()\n    files = [f for f in api.list_repo_tree(REPO, path_in_repo=\'FL2VA\', recursive=True, revision=REVISION)\n             if hasattr(f, \'size\') and f.size > 0]\n    metadata = {f.path: {\'size\': f.size, \'sha256\': f.lfs.sha256 if f.lfs else None} for f in files}\n    total = sum(f[\'size\'] for f in metadata.values())\n    missing = sum(f[\'size\'] for name, f in metadata.items()\n                  if not (local/name).is_file() or (local/name).stat().st_size != f[\'size\'])\n    if shutil.disk_usage(local).free < missing + 8 * 1024**3:\n        raise RuntimeError(f\'Need {missing / 1024**3:.1f} GiB for weights plus 8 GiB free headroom\')\n    print(f\'Pinned {REPO}@{REVISION}; FL2VA only: {total / 1024**3:.2f} GiB\', flush=True)\n    print(\'Drive free space is not inferred from FUSE statvfs; a quota error fails this backup.\', flush=True)\n    drive_manifest = drive/\'verified-manifest.json\'\n    prior = json.loads(drive_manifest.read_text()) if drive_manifest.is_file() else {}\n    verified = prior.get(\'files\', {}) if prior.get(\'revision\') == REVISION else {}\n    restore = [name for name, meta in metadata.items()\n               if verified.get(name, {}).get(\'size\') == meta[\'size\']\n               and (drive/name).is_file() and (drive/name).stat().st_size == meta[\'size\']\n               and (not (local/name).is_file() or (local/name).stat().st_size != meta[\'size\'])]\n    if restore:\n        print(f\'Restoring {len(restore)} cached files from Drive to local SSD\', flush=True)\n        def restore_one(name):\n            meta = metadata[name]\n            try:\n                copy_verified(drive/name, local/name, meta[\'size\'], meta[\'sha256\'] or verified[name][\'sha256\'])\n                print(\'Restored \' + name, flush=True)\n            except ValueError:\n                verified.pop(name, None)\n                print(\'Drive checksum mismatch; will download a fresh official copy: \' + name, flush=True)\n        with ThreadPoolExecutor(max_workers=2) as pool:\n            list(pool.map(restore_one, restore))\n    download = [name for name, meta in metadata.items()\n                if not (local/name).is_file() or (local/name).stat().st_size != meta[\'size\']]\n    if download:\n        snapshot_download(REPO, revision=REVISION, allow_patterns=download,\n                          local_dir=str(local), max_workers=4)\n    # Keep the model identity for SGLang pipeline discovery without downloading duplicate root weights.\n    snapshot_download(REPO, revision=REVISION, allow_patterns=[\'model_index.json\', \'LICENSE\'],\n                      local_dir=str(local), max_workers=2)\n    print(\'MODEL_READY_ON_LOCAL_SSD: \' + str(local), flush=True)\n    result = {}\n    for name, meta in metadata.items():\n        if (drive/name).is_file() and (drive/name).stat().st_size == meta[\'size\'] and verified.get(name, {}).get(\'sha256\'):\n            result[name] = verified[name]\n            continue\n        try:\n            sha = copy_verified(local/name, drive/name, meta[\'size\'], meta[\'sha256\'])\n        except ValueError:\n            hf_hub_download(REPO, name, revision=REVISION, local_dir=str(local), force_download=True)\n            sha = copy_verified(local/name, drive/name, meta[\'size\'], meta[\'sha256\'])\n        result[name] = {\'size\': meta[\'size\'], \'sha256\': sha}\n        record = {\'repo\': REPO, \'revision\': REVISION, \'files\': {**verified, **result}, \'complete\': False}\n        temp = drive_manifest.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(record, indent=2)); temp.replace(drive_manifest)\n        print(\'Verified Drive backup \' + name, flush=True)\n    for name in [\'model_index.json\', \'LICENSE\']:\n        source = local/name\n        copy_verified(source, drive/name, source.stat().st_size)\n    record = {\'repo\': REPO, \'revision\': REVISION, \'bytes\': total, \'files\': result, \'complete\': True}\n    for folder in [local, drive]:\n        manifest = folder/\'verified-manifest.json\'\n        temp = manifest.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(record, indent=2)); temp.replace(manifest)\n    print(\'DRIVE_BACKUP_COMPLETE: \' + str(drive), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--local\', type=Path, default=Path(\'/content/MiniMax-H3\'))\n    parser.add_argument(\'--drive\', type=Path, default=Path(\'/content/drive/MyDrive/TapNowLocal/models/MiniMax-H3\')/REVISION)\n    parser.add_argument(\'--self-test\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.self_test:\n        with tempfile.TemporaryDirectory() as folder:\n            src, dst = Path(folder)/\'src\', Path(folder)/\'dst\'\n            src.write_bytes(b\'weights\')\n            digest = hashlib.sha256(b\'weights\').hexdigest()\n            assert copy_verified(src, dst, 7, digest) == digest\n            assert dst.read_bytes() == b\'weights\'\n            try:\n                copy_verified(src, dst, 8, digest)\n                raise AssertionError(\'Expected integrity error\')\n            except ValueError:\n                assert dst.read_bytes() == b\'weights\'\n                assert not dst.with_name(\'dst.part\').exists()\n        print(\'model cache integrity checks passed\')\n    else:\n        os.environ[\'HF_XET_CHUNK_CACHE_SIZE_BYTES\'] = \'0\'\n        cache(args.local, args.drive)\n', 'auth_proxy.py': '"""Expose only authenticated generation routes; keep SGLang management private."""\nimport hmac\nimport os\nimport re\n\n\ndef allowed_route(method, path):\n    return (method == \'GET\' and path in (\'/health\', \'/v1/models\')) or (\n        method == \'POST\' and path == \'/v1/videos\'\n    ) or (method == \'GET\' and re.fullmatch(r\'/v1/videos/[A-Za-z0-9_-]{1,100}(?:/content)?\', path) is not None)\n\n\ndef authorized(header, token):\n    return bool(token) and hmac.compare_digest(header.encode(), (\'Bearer \' + token).encode())\n\n\ndef create_app():\n    import httpx\n    from contextlib import asynccontextmanager\n    from fastapi import FastAPI, Request\n    from fastapi.responses import JSONResponse, StreamingResponse\n\n    token = os.environ.get(\'H3_PROXY_TOKEN\', \'\')\n    if len(token) < 32:\n        raise RuntimeError(\'Set H3_PROXY_TOKEN to a random token of at least 32 characters\')\n\n    @asynccontextmanager\n    async def lifespan(app):\n        async with httpx.AsyncClient(base_url=\'http://127.0.0.1:30020\', timeout=60, trust_env=False) as client:\n            app.state.upstream = client\n            yield\n\n    app = FastAPI(lifespan=lifespan, docs_url=None, redoc_url=None, openapi_url=None)\n\n    @app.api_route(\'/{route:path}\', methods=[\'GET\', \'POST\', \'PUT\', \'PATCH\', \'DELETE\', \'OPTIONS\'])\n    async def proxy(route: str, request: Request):\n        if not authorized(request.headers.get(\'authorization\', \'\'), token):\n            return JSONResponse({\'error\': \'Unauthorized\'}, status_code=401)\n        path = \'/\' + route\n        if not allowed_route(request.method, path):\n            return JSONResponse({\'error\': \'Route not exposed\'}, status_code=404)\n        body = bytearray()\n        async for chunk in request.stream():\n            body.extend(chunk)\n            if len(body) > 1024 * 1024:\n                return JSONResponse({\'error\': \'Request too large\'}, status_code=413)\n        try:\n            upstream_request = request.app.state.upstream.build_request(\n                request.method, path, content=bytes(body),\n                headers={\'Content-Type\': request.headers.get(\'content-type\', \'application/json\'),\n                         \'Accept-Encoding\': \'identity\'},\n            )\n            upstream = await request.app.state.upstream.send(upstream_request, stream=True)\n        except httpx.HTTPError:\n            return JSONResponse({\'error\': \'Model service unavailable\'}, status_code=503)\n        headers = {key: value for key, value in upstream.headers.items()\n                   if key in (\'content-type\', \'content-length\', \'content-disposition\', \'content-encoding\')}\n        async def stream():\n            try:\n                async for chunk in upstream.aiter_raw():\n                    yield chunk\n            finally:\n                await upstream.aclose()\n        return StreamingResponse(stream(), status_code=upstream.status_code, headers=headers)\n\n    return app\n\n\nif __name__ == \'__main__\':\n    import sys\n    if \'--self-test\' in sys.argv:\n        assert allowed_route(\'GET\', \'/v1/videos/video_123/content\')\n        assert allowed_route(\'POST\', \'/v1/videos\')\n        for method, path in [(\'DELETE\', \'/v1/videos/x\'), (\'GET\', \'/server_info\'),\n                             (\'GET\', \'/v1/videos/../../server_info\'), (\'POST\', \'/update_weights\')]:\n            assert not allowed_route(method, path)\n        assert authorized(\'Bearer \' + \'x\' * 32, \'x\' * 32)\n        assert not authorized(\'Bearer wrong\', \'x\' * 32)\n        assert not authorized(\'Bearer \', \'\')\n        print(\'proxy policy checks passed\')\n    else:\n        import uvicorn\n        uvicorn.run(create_app(), host=\'127.0.0.1\', port=30011, access_log=False)\n', 'start_model.sh': "#!/usr/bin/env bash\nset -euo pipefail\ntest -f /content/h3-install.ok || { echo 'Install must finish first'; exit 1; }\ntest -f /content/MiniMax-H3/FL2VA/model_index.json || { echo 'Native FL2VA weights missing'; exit 1; }\ncommand -v ffmpeg >/dev/null\ncommand -v ffprobe >/dev/null\nmkdir -p /content/h3-videos\n# shortcut: unbenchmarked 96 GB single-card placement; tune only after a real baseline.\nexec /content/h3-venv/bin/sglang serve \\\n  --model-type diffusion \\\n  --model-path /content/MiniMax-H3 \\\n  --model-id MiniMax-H3 \\\n  --pipeline-class-name MiniMaxH3Pipeline \\\n  --served-model-name MiniMaxAI/MiniMax-H3 \\\n  --model-variant fl2va --backend sglang \\\n  --num-gpus 1 --tp-size 1 --ulysses-degree 1 \\\n  --performance-mode manual \\\n  --component-residency dit=component-offload text_encoder=component-offload vae=component-offload audio_vae=resident \\\n  --pin-cpu-memory false --enable-torch-compile false \\\n  --attention-backend fa --warmup-mode off \\\n  --output-path /content/h3-videos \\\n  --host 127.0.0.1 --port 30020\n", 'open_tunnel.py': '"""Run only after explicit approval to expose the authenticated video proxy."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport urllib.request\n\n\ndef main():\n    if os.environ.get(\'H3_ALLOW_PUBLIC_TUNNEL\') != \'yes\':\n        raise SystemExit(\'Public exposure is disabled. Confirm the temporary authenticated tunnel first.\')\n    request = urllib.request.Request(\'http://127.0.0.1:30011/health\', headers={\n        \'Authorization\': \'Bearer \' + os.environ[\'H3_PROXY_TOKEN\']})\n    with urllib.request.urlopen(request, timeout=10) as response:\n        if response.status != 200:\n            raise RuntimeError(\'Authenticated proxy is not ready\')\n    with urllib.request.urlopen(\'https://api.github.com/repos/cloudflare/cloudflared/releases/latest\') as response:\n        release = json.load(response)\n    asset = next(a for a in release[\'assets\'] if a[\'name\'] == \'cloudflared-linux-amd64\')\n    digest = asset.get(\'digest\', \'\')\n    if not digest.startswith(\'sha256:\'):\n        raise RuntimeError(\'Official release does not provide a SHA256 digest; verify the binary manually\')\n    binary = Path(\'/content/cloudflared\')\n    temp = binary.with_suffix(\'.part\')\n    hasher = hashlib.sha256()\n    try:\n        with urllib.request.urlopen(asset[\'browser_download_url\']) as src, temp.open(\'wb\') as dst:\n            while chunk := src.read(1024 * 1024):\n                hasher.update(chunk); dst.write(chunk)\n        if hasher.hexdigest() != digest[7:]:\n            raise RuntimeError(\'Cloudflare release checksum mismatch\')\n        temp.chmod(0o700); temp.replace(binary)\n    finally:\n        temp.unlink(missing_ok=True)\n    Path(\'/content/h3-cloudflared-version.json\').write_text(json.dumps({\n        \'version\': release[\'tag_name\'], \'sha256\': digest[7:]}))\n    subprocess.run([str(binary), \'tunnel\', \'--url\', \'http://127.0.0.1:30011\', \'--no-autoupdate\'], check=True)\n\n\nif __name__ == \'__main__\':\n    import sys\n    if \'--self-test\' in sys.argv:\n        from unittest.mock import patch\n        with patch.dict(os.environ, {\'H3_ALLOW_PUBLIC_TUNNEL\': \'no\'}), patch(\'urllib.request.urlopen\', side_effect=AssertionError(\'Must not use network\')):\n            try:\n                main()\n                raise AssertionError(\'Tunnel must stay closed without approval\')\n            except SystemExit as error:\n                assert \'disabled\' in str(error)\n        print(\'tunnel fail-closed check passed (no network exposure)\')\n    else:\n        main()\n'}
h3_dir = pathlib.Path('/content/tapnow-h3')
h3_dir.mkdir(exist_ok=True)
for name, source in h3_files.items():
    (h3_dir/name).write_text(source)
print('Deployment helpers written:', ', '.join(h3_files))

后台安装与模型下载可并行。安装使用固定 SGLang 提交及 Python 3.12 独立环境；保留现有 Notebook Python 环境。关闭可选 SRT Rust 扩展是官方支持的构建开关。

In [ ]:
h3_install = subprocess.Popen(['bash',str(h3_dir/'install.sh')], stdout=open('/content/h3-install.log','w'),stderr=subprocess.STDOUT)
print('Install PID:', h3_install.pid)

In [ ]:
h3_download = subprocess.Popen(['uv','run','--with','huggingface-hub','--with','hf-xet','python',str(h3_dir/'cache_model.py')],stdout=open('/content/h3-download.log','w'),stderr=subprocess.STDOUT,env={**os.environ,'UV_NO_CACHE':'1','HF_XET_CHUNK_CACHE_SIZE_BYTES':'0'})
print('Download + Drive backup PID:', h3_download.pid)

读取进度。出现 `MODEL_READY_ON_LOCAL_SSD` 后可启动模型，Drive 备份继续进行。必须出现 `DRIVE_BACKUP_COMPLETE` 才代表可供下次复用的完整校验副本已保存。

In [ ]:
for name in ('install','download'):
    log=pathlib.Path('/content/h3-'+name+'.log')
    print('\n'+name.upper()+'\n'+(log.read_text()[-1800:] if log.exists() else 'not started'))

安装完成、权重本地就绪后再运行。服务仅监听 Colab 本机。此保守单卡配置按阶段整体移动组件，DiT 的 50 次采样不会逐层读取 Drive；它不等于经过测速的最优方案。

In [ ]:
assert pathlib.Path('/content/h3-install.ok').exists(), 'Installation has not finished'
assert 'MODEL_READY_ON_LOCAL_SSD' in pathlib.Path('/content/h3-download.log').read_text(), 'Weights are not ready'
h3_model = subprocess.Popen(['bash',str(h3_dir/'start_model.sh')],stdout=open('/content/h3-model.log','w'),stderr=subprocess.STDOUT)
print('Model PID:',h3_model.pid)

In [ ]:
print(pathlib.Path('/content/h3-model.log').read_text()[-2500:])
import urllib.request
try:
    print(urllib.request.urlopen('http://127.0.0.1:30020/health',timeout=5).read().decode())
except Exception as error:
    print('Not ready:',type(error).__name__)

标准冷启动首测，实际会消耗 GPU。完成后记录 MP4 与 JSON；API 计时终点是文件下载结束，画布端另记录视频可播放时刻。

In [ ]:
h3_bench = subprocess.Popen([sys.executable,str(h3_dir/'benchmark.py'),'--label','cold-standard-4s-768p'],stdout=open('/content/h3-benchmark.log','w'),stderr=subprocess.STDOUT)
print('Benchmark PID:',h3_bench.pid)

In [ ]:
print(pathlib.Path('/content/h3-benchmark.log').read_text()[-2000:])
result=pathlib.Path('/content/h3-benchmarks/cold-standard-4s-768p.json')
if result.exists():
    print(json.loads(result.read_text())['totalMs'],'ms')
    print(subprocess.check_output(['ffprobe','-v','error','-show_entries','stream=codec_type,codec_name,width,height,r_frame_rate,duration','-show_entries','format=duration','-of','json','/content/h3-benchmarks/cold-standard-4s-768p.mp4'],text=True))

重复请求测速。同一提示词会受 SGLang 条件编码缓存影响，因此明确标记为重复输入；不能把这个数直接当所有新提示词的速度。

In [ ]:
h3_warm = subprocess.Popen([sys.executable,str(h3_dir/'benchmark.py'),'--label','warm-repeated-input-4s-768p'],stdout=open('/content/h3-warm-benchmark.log','w'),stderr=subprocess.STDOUT)
print('Warm benchmark PID:',h3_warm.pid)

启动仅允许视频 API 的认证代理。输入至少 32 字符的一次性 Token，并在本地画布的「模型连接」中使用同一个 Token。Token 不写入 Drive。

In [ ]:
import getpass
h3_token=getpass.getpass('Temporary H3 Token (32+ characters): ')
assert len(h3_token)>=32
h3_proxy_env={**os.environ,'H3_PROXY_TOKEN':h3_token}
h3_proxy=subprocess.Popen(['/content/h3-venv/bin/python',str(h3_dir/'auth_proxy.py')],env=h3_proxy_env,stdout=open('/content/h3-proxy.log','w'),stderr=subprocess.STDOUT)
print('Proxy PID:',h3_proxy.pid)

**公开访问步骤默认禁用。** 下面只将带 Bearer 认证的代理通过临时 HTTPS 隧道提供给本地画布；不开放 SGLang 管理端口。自动助手执行前需要获得这一次公开访问的明确确认。确认后把 `False` 改为 `True`。临时域名在重启后会变化。

In [ ]:
h3_allow_public_tunnel = False
assert h3_allow_public_tunnel, 'Confirm this temporary public HTTPS tunnel before enabling it'
h3_tunnel=subprocess.Popen([sys.executable,str(h3_dir/'open_tunnel.py')],env={**h3_proxy_env,'H3_ALLOW_PUBLIC_TUNNEL':'yes'},stdout=open('/content/h3-tunnel.log','w'),stderr=subprocess.STDOUT)
print('Tunnel PID:',h3_tunnel.pid)

In [ ]:
import re
h3_tunnel_log=pathlib.Path('/content/h3-tunnel.log').read_text()
h3_urls=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',h3_tunnel_log)
print(h3_urls[-1] if h3_urls else h3_tunnel_log[-1200:])

把脚本、环境信息和实测结果备份到 Drive。这里只复制本任务文件，不复制 Token 或隧道配置。Notebook 自身可通过 Colab 文件菜单保存到同一 Drive 文件夹。

In [ ]:
h3_backup=pathlib.Path('/content/drive/MyDrive/TapNowLocal/deployment')
h3_backup.mkdir(parents=True,exist_ok=True)
for name in h3_files:
    shutil.copy2(h3_dir/name,h3_backup/name)
for name in ('h3-environment.txt','h3-cloudflared-version.json'):
    path=pathlib.Path('/content')/name
    if path.exists(): shutil.copy2(path,h3_backup/name)
benchmarks=pathlib.Path('/content/h3-benchmarks')
if benchmarks.exists(): shutil.copytree(benchmarks,h3_backup/'benchmarks',dirs_exist_ok=True)
print('Saved deployment and available benchmark files:',h3_backup)

结束当前会话时，先停止隧道，再停止本任务启动的代理与模型进程；不要终止其他 Notebook 的进程。Drive 模型缓存保留。

In [ ]:
for process_name in ('h3_tunnel','h3_proxy','h3_model'):
    process=globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        print('Stopped',process_name)